<a href="https://colab.research.google.com/github/MoussaBane/extented-named-entity-recognation-ener/blob/update_03/MB_MasterProjet_ener_kfold_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Turkish ENER — 5-model k-fold suite (Colab GPU)

Bu defter, konferans bildirisinde CRF için yapılan 10-fold analizin aynısını
(**per-entity-type precision/recall/F1 + sparse confusion matrix**) kalan 5
model ailesi için üretir: **BERT fine-tuned, Attention-NER (frozen/fine-tuned),
Character-BERT, Contrastive-NER**.

**Önce yapman gerekenler:**
1. Üstteki menüden **Çalışma zamanı (Runtime) → Çalışma zamanı türünü değiştir → T4 GPU** seç.
2. Bu notebook'u çalıştırmadan önce, sohbette paylaşılan şu 2 dosyayı bilgisayarına indir (isim değiştirme):
   - `run_char_ner_PATCHED.py`
   - `run_contrastive_ner_PATCHED.py`
   (Bunlar, per-label metrik/confusion matrix kaydetmeyi eklediğim yamalı sürümler.)
3. Aşağıdaki hücreleri sırayla çalıştır; 4. hücrede bu 2 dosyayı yükleman istenecek.

### 1) GPU kontrolü

In [ ]:
!nvidia-smi

Sun Aug  9 19:44:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 2) Repoyu klonla ve bağımlılıkları kur

In [ ]:
!git clone -b update_03 https://github.com/MoussaBane/extented-named-entity-recognation-ener.git repo
%cd repo
!pip install -q torch transformers sklearn-crfsuite seqeval pandas evaluate

Cloning into 'repo'...
remote: Enumerating objects: 608, done.
remote: Counting objects: 100% (608/608), done.
remote: Compressing objects: 100% (410/410), done.
remote: Total 608 (delta 160), reused 541 (delta 106), pack-reused 0 (from 0)
Receiving objects: 100% (608/608), 6.72 MiB | 30.32 MiB/s, done.
Resolving deltas: 100% (160/160), done.
Filtering content: 100% (9/9), 1.23 GiB | 17.59 MiB/s, done.
/content/repo
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.7 MB/s eta 0:00:00


### 3) Google Drive'ı bağla (önerilir)
Colab oturumu koparsa sonuçları kaybetmemek için sonuçları doğrudan Drive'a yazacağız.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/ener_kfold_results"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print("Sonuçlar buraya yazılacak:", DRIVE_RESULTS_DIR)

Mounted at /content/drive
Sonuçlar buraya yazılacak: /content/drive/MyDrive/ener_kfold_results


### 4) Yamalı script dosyalarını yükle
Az önce indirdiğin `run_char_ner_PATCHED.py` ve `run_contrastive_ner_PATCHED.py` dosyalarını seç.

In [ ]:
from google.colab import files
uploaded = files.upload()  # run_char_ner_PATCHED.py ve run_contrastive_ner_PATCHED.py seç

import shutil
if "run_char_ner_PATCHED.py" in uploaded:
    shutil.move("run_char_ner_PATCHED.py", "scripts/run_char_ner.py")
    print("scripts/run_char_ner.py güncellendi (patched)")
if "run_contrastive_ner_PATCHED.py" in uploaded:
    shutil.move("run_contrastive_ner_PATCHED.py", "scripts/run_contrastive_ner.py")
    print("scripts/run_contrastive_ner.py güncellendi (patched)")

Saving run_contrastive_ner_PATCHED.py to run_contrastive_ner_PATCHED.py
Saving run_char_ner_PATCHED.py to run_char_ner_PATCHED.py
scripts/run_char_ner.py güncellendi (patched)
scripts/run_contrastive_ner.py güncellendi (patched)


### 5) `make_kfold_conll.py` script'ini oluştur ve fold dosyalarını üret

In [ ]:
%%writefile scripts/make_kfold_conll.py
"""Split a CoNLL-BIO file into N folds train/eval files, using the SAME
seeded permutation logic as run_crf_baseline.py and run_cross_validation.py,
so that fold boundaries are identical across all model families.
"""
import argparse
import os
import numpy as np

from ner_stats.data_utils import read_conll_bio


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data-file", default="data/full_train.conll")
    p.add_argument("--output-dir", default="data/kfold_10")
    p.add_argument("--num-folds", type=int, default=10)
    p.add_argument("--seed", type=int, default=42)
    return p.parse_args()


def make_fold_indices(n_samples, n_folds, seed):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(n_samples)
    return np.array_split(perm, n_folds)


def write_conll(path, tokens, labels):
    with open(path, "w", encoding="utf-8") as f:
        for sent_tok, sent_lab in zip(tokens, labels):
            for t, l in zip(sent_tok, sent_lab):
                f.write(f"{t}\t{l}\n")
            f.write("\n")


def main():
    args = parse_args()
    tokens, labels = read_conll_bio(args.data_file)
    n = len(tokens)
    print(f"Loaded {n} sentences from {args.data_file}")

    folds = make_fold_indices(n, args.num_folds, args.seed)
    os.makedirs(args.output_dir, exist_ok=True)

    for i in range(args.num_folds):
        val_idx = folds[i]
        train_idx = [idx for j in range(args.num_folds) for idx in folds[j] if j != i]

        train_tokens = [tokens[k] for k in train_idx]
        train_labels = [labels[k] for k in train_idx]
        val_tokens = [tokens[k] for k in val_idx]
        val_labels = [labels[k] for k in val_idx]

        train_path = os.path.join(args.output_dir, f"fold_{i}_train.conll")
        eval_path = os.path.join(args.output_dir, f"fold_{i}_eval.conll")
        write_conll(train_path, train_tokens, train_labels)
        write_conll(eval_path, val_tokens, val_labels)
        print(f"fold {i}: {len(train_idx)} train / {len(val_idx)} eval")

    print("Done.")


if __name__ == "__main__":
    main()

Writing scripts/make_kfold_conll.py


In [ ]:
NUM_FOLDS = 4  # 4 => mevcut bildirideki sonuçlarla dogrudan karsilastirilabilir. Zaman varsa 10 yap.
SEED = 42

# Construct the command string using Python f-string interpolation
command = f"""export PYTHONPATH=$(pwd):$PYTHONPATH && python3 scripts/make_kfold_conll.py \
  --data-file data/full_train.conll \
  --output-dir data/kfold_{NUM_FOLDS} \
  --num-folds {NUM_FOLDS} \
  --seed {SEED}"""

# Execute the fully interpolated command string
!{command}

Loaded 1035 sentences from data/full_train.conll
fold 0: 776 train / 259 eval
fold 1: 776 train / 259 eval
fold 2: 776 train / 259 eval
fold 3: 777 train / 258 eval
Done.


### 6) BERT fine-tuned (kendi içinde k-fold döngüsü var, tek çağrı yeterli)
`Trainer` GPU'yu otomatik kullanır, `--device` bayrağı gerekmez.

In [ ]:
import os
bert_out = f"/content/drive/MyDrive/ener_kfold_results/bert_{NUM_FOLDS}fold"
if not os.path.exists(os.path.join(bert_out, "crf_cv_summary.json")) and not os.path.exists(bert_out):
    # Set PYTHONPATH to include the current directory (/content/repo) so ner_stats can be found
    command = f"""export PYTHONPATH=$(pwd):$PYTHONPATH && python3 scripts/run_cross_validation.py \
      --data-file data/full_train.conll \
      --output-dir {bert_out} \
      --num-folds {NUM_FOLDS} \
      --seed {SEED}"""
    !{command}
else:
    print("BERT sonuçları zaten var, atlanıyor:", bert_out)

config.json: 100% 385/385 [00:00<00:00, 1.33MB/s]
tokenizer_config.json: 100% 60.0/60.0 [00:00<00:00, 210kB/s]
vocab.txt: 100% 251k/251k [00:00<00:00, 13.0MB/s]

model.safetensors: downloading bytes:  37% 166M/445M [00:02<00:02, 106MB/s, 13.7MB/s  ] 
model.safetensors: downloading bytes:  45% 201M/445M [00:02<00:02, 112MB/s, 16.8MB/s  ]
model.safetensors: downloading bytes:  48% 215M/445M [00:02<00:02, 95.5MB/s, 18.7MB/s  ]
model.safetensors: downloading bytes:  59% 262M/445M [00:03<00:01, 110MB/s, 21.5MB/s  ]
model.safetensors: downloading bytes:  73% 324M/445M [00:03<00:00, 170MB/s, 25.6MB/s  ]
model.safetensors: downloading bytes:  78% 346M/445M [00:03<00:00, 182MB/s, 27.6MB/s  ]
model.safetensors: downloading bytes:  87% 386M/445M [00:03<00:00, 128MB/s, 31.1MB/s  ]
model.safetensors: downloading bytes:  95% 421M/445M [00:04<00:00, 145MB/s, 33.3MB/s  ]
model.safetensors: reconstructing file:  90% 402M/445M [00:04<00:00, 137MB/s, 30.9MB/s  ]
model.safetensors: downloading bytes: 100%

In [ ]:
import os, zipfile

SRC = "/content/drive/MyDrive/ener_kfold_results/bert_4fold"
OUT_ZIP = "/content/bert_4fold_full_metrics.zip"

KEEP_SUFFIXES = (
    "per_class_metrics.csv", "confusion_matrix.csv", "confusion_matrix_top20.csv",
    "classification_report.json", "metrics.json", "metrics_summary.csv", "cv_summary.json",
)

count = 0
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(SRC):
        if "model" in dirs:
            dirs.remove("model")
        for f in files:
            if f.endswith(KEEP_SUFFIXES):
                zf.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), SRC))
                count += 1

print(f"{count} dosya eklendi")
from google.colab import files
files.download(OUT_ZIP)

25 dosya eklendi


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 7) Attention-NER, Character-BERT, Contrastive-NER — fold döngüsü
Her model `--device cuda` ile çalıştırılır. Bir fold zaten tamamlanmışsa (Drive'da klasörü varsa) atlanır — Colab oturumu koparsa kaldığın yerden devam edebilirsin, hücreyi tekrar çalıştırman yeterli.

In [ ]:
import os, subprocess, sys

RESULTS_ROOT = f"/content/drive/MyDrive/ener_kfold_results/kfold_{NUM_FOLDS}"
KFOLD_DIR = f"data/kfold_{NUM_FOLDS}"

def run(cmd):
    print(">>>", " ".join(cmd))
    # Add the current directory to PYTHONPATH for the subprocess
    env = os.environ.copy()
    current_repo_path = os.getcwd() # Should be /content/repo
    if "PYTHONPATH" in env:
        env["PYTHONPATH"] = f"{current_repo_path}:{env["PYTHONPATH"]}"
    else:
        env["PYTHONPATH"] = current_repo_path

    try:
        # Capture stdout and stderr from the subprocess to diagnose potential errors
        result = subprocess.run(cmd, check=True, env=env, capture_output=True, text=True)
        print(result.stdout)
        if result.stderr:
            print(result.stderr, file=sys.stderr)
    except subprocess.CalledProcessError as e:
        print(f"Subprocess failed with exit code {e.returncode}", file=sys.stderr)
        if e.stdout:
            print("Subprocess stdout:\n", e.stdout, file=sys.stderr)
        if e.stderr:
            print("Subprocess stderr:\n", e.stderr, file=sys.stderr)
        raise # Re-raise the exception after printing details

def already_done(out_dir):
    return os.path.exists(os.path.join(out_dir, "per_class_metrics.csv")) or \
           os.path.exists(os.path.join(out_dir, "per_label_report.csv"))

for i in range(NUM_FOLDS):
    train = f"{KFOLD_DIR}/fold_{i}_train.conll"
    ev = f"{KFOLD_DIR}/fold_{i}_eval.conll"
    print(f"\n========== FOLD {i}/{NUM_FOLDS-1} ==========")

    # Attention-NER, fine-tuned
    out = f"{RESULTS_ROOT}/attention_ner_finetuned/fold_{i}"
    if not already_done(out):
        run(["python3", "scripts/run_attention_ner.py",
             "--train-file", train, "--eval-file", ev,
             "--output-dir", out, "--num-train-epochs", "5",
             "--seed", str(SEED), "--device", "cuda"])
    else:
        print("skip (already done):", out)

    # Attention-NER, frozen BERT
    out = f"{RESULTS_ROOT}/attention_ner_frozen/fold_{i}"
    if not already_done(out):
        run(["python3", "scripts/run_attention_ner.py",
             "--train-file", train, "--eval-file", ev,
             "--output-dir", out, "--num-train-epochs", "5",
             "--freeze-bert", "--seed", str(SEED), "--device", "cuda"])
    else:
        print("skip (already done):", out)

    # Character-BERT hybrid
    out = f"{RESULTS_ROOT}/char_ner/fold_{i}"
    if not already_done(out):
        run(["python3", "scripts/run_char_ner.py",
             "--train-file", train, "--eval-file", ev,
             "--output-dir", out, "--num-epochs", "3",
             "--seed", str(SEED), "--device", "cuda"])
    else:
        print("skip (already done):", out)

    # Contrastive-NER
    out = f"{RESULTS_ROOT}/contrastive_ner/fold_{i}"
    if not already_done(out):
        run(["python3", "scripts/run_contrastive_ner.py",
             "--train-file", train, "--eval-file", ev,
             "--output-dir", out, "--num-epochs", "3",
             "--seed", str(SEED), "--device", "cuda"])
    else:
        print("skip (already done):", out)

print("\nTüm foldlar tamamlandı. Sonuçlar Drive'da:", RESULTS_ROOT)


========== FOLD 0/3 ==========
>>> python3 scripts/run_attention_ner.py --train-file data/kfold_4/fold_0_train.conll --eval-file data/kfold_4/fold_0_eval.conll --output-dir /content/drive/MyDrive/ener_kfold_results/kfold_4/attention_ner_finetuned/fold_0 --num-train-epochs 5 --seed 42 --device cuda
[INFO] Training AttentionNERModel | freeze_bert=False | d_head=256 | epochs=5 | steps=485 | o_weight=1.0
  Epoch 1/5 — avg loss: 2.5988
  Epoch 2/5 — avg loss: 1.2552
  Epoch 3/5 — avg loss: 1.0100
  Epoch 4/5 — avg loss: 0.8806
  Epoch 5/5 — avg loss: 0.7989
[INFO] Model saved to /content/drive/MyDrive/ener_kfold_results/kfold_4/attention_ner_finetuned/fold_0/model
[INFO] AttentionNER macro-F1 (w/o O): 0.0971 | accuracy: 0.8149
[INFO] Results saved to /content/drive/MyDrive/ener_kfold_results/kfold_4/attention_ner_finetuned/fold_0

>>> python3 scripts/run_attention_ner.py --train-file data/kfold_4/fold_0_train.conll --eval-file data/kfold_4/fold_0_eval.conll --output-dir /content/drive/MyDr


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20073.27it/s]
[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[INFO] Training AttentionNERModel | freeze_bert=True | d_head=256 | epochs=5 | steps=485 | o_weight=1.0
  Epoch 1/5 — avg loss: 4.4413
  Epoch 2/5 — avg loss: 1.8710
  Epoch 3/5 — avg loss: 1.7020
  Epoch 4/5 — avg loss: 1.6617
  Epoch 5/5 — avg loss: 1.6454
[INFO] Model saved to /content/drive/MyDrive/ener_kfold_results/kfold_4/attention_ner_frozen/fold_0/model
[INFO] AttentionNER macro-F1 (w/o O): 0.0000 | accuracy: 0.7620
[INFO] Results saved to /content/drive/MyDrive/ener_kfold_results/kfold_4/attention_ner_frozen/fold_0

>>> python3 scripts/run_char_ner.py --train-file data/kfold_4/fold_0_train.conll --eval-file data/kfold_4/fold_0_eval.conll --output-dir /content/drive/MyDrive/ener_kfold_results/kfold_4/char_ner/fold_0 --num-epochs 3 --seed 42 --device cuda



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14222.34it/s]
[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[INFO] Epoch 1/3  loss=2.9306  macro-F1=0.0000  acc=0.0000
[INFO] Epoch 2/3  loss=1.3644  macro-F1=0.0296  acc=0.5841
[INFO] Epoch 3/3  loss=1.1354  macro-F1=0.0413  acc=0.5639

[INFO] CharBERT — final macro-F1 (w/o O): 0.0413
[INFO] Results saved to /content/drive/MyDrive/ener_kfold_results/kfold_4/char_ner/fold_0

>>> python3 scripts/run_contrastive_ner.py --train-file data/kfold_4/fold_0_train.conll --eval-file data/kfold_4/fold_0_eval.conll --output-dir /content/drive/MyDrive/ener_kfold_results/kfold_4/contrastive_ner/fold_0 --num-epochs 3 --seed 42 --device cuda


...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4628.14it/s]
[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[INFO] Epoch 1/3  loss=2.8081  macro-F1=0.0091  acc=0.6000
[INFO] Epoch 2/3  loss=1.2656  macro-F1=0.0474  acc=0.6716
[INFO] Epoch 3/3  loss=1.0607  macro-F1=0.0621  acc=0.5586

[INFO] CharBERT — final macro-F1 (w/o O): 0.0621
[INFO] Results saved to /content/drive/MyDrive/ener_kfold_results/kfold_4/char_ner/fold_3

>>> python3 scripts/run_contrastive_ner.py --train-file data/kfold_4/fold_3_train.conll --eval-file data/kfold_4/fold_3_eval.conll --output-dir /content/drive/MyDrive/ener_kfold_results/kfold_4/contrastive_ner/fold_3 --num-epochs 3 --seed 42 --device cuda



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5314.11it/s]
[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[INFO] Epoch 1/3  total=3.0959  CE=2.8504  SupCon=5.3051  macro-F1=0.0080  acc=0.5823
[INFO] Epoch 2/3  total=1.6427  CE=1.2716  SupCon=4.9833  macro-F1=0.0387  acc=0.6638
[INFO] Epoch 3/3  total=1.4580  CE=1.0776  SupCon=4.8815  macro-F1=0.0429  acc=0.5749

[INFO] ContrastiveBERT — final macro-F1 (w/o O): 0.0429
[INFO] Results saved to /content/drive/MyDrive/ener_kfold_results/kfold_4/contrastive_ner/fold_3
[INFO] BERT backbone saved to /content/drive/MyDrive/ener_kfold_results/kfold_4/contrastive_ner/fold_3/bert_backbone (re-usable with --skip-bert-train).


Tüm foldlar tamamlandı. Sonuçlar Drive'da: /content/drive/MyDrive/ener_kfold_results/kfold_4



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19855.99it/s]
[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.46s/it]



7b) CVA (one-shot prototype classifier) — fold döngüsü
Eğitim gerektirmez (sadece BERT embedding + cosine similarity), bu yüzden en hızlı adım. Aynı --device cuda ve fold dosyalarını kullanır; tamamlanan foldlar atlanır.

In [ ]:
from google.colab import files
uploaded = files.upload()  # compare_context_cva_PATCHED.py seç

import shutil
shutil.move("compare_context_cva_PATCHED.py", "scripts/compare_context_cva.py")
print("scripts/compare_context_cva.py güncellendi (patched)")

Saving compare_context_cva_PATCHED.py to compare_context_cva_PATCHED.py
scripts/compare_context_cva.py güncellendi (patched)


In [ ]:
import os, subprocess

CVA_ROOT = f"/content/drive/MyDrive/ener_kfold_results/kfold_{NUM_FOLDS}/cva"
KFOLD_DIR = f"data/kfold_{NUM_FOLDS}"

def run(cmd):
    print(">>>", " ".join(cmd))
    env = os.environ.copy()
    env["PYTHONPATH"] = os.getcwd() + os.pathsep + env.get("PYTHONPATH", "")
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print("----- STDERR -----")
        print(result.stderr[-3000:])
        raise RuntimeError(f"Command failed with code {result.returncode}")

def cva_already_done(out_dir):
    return os.path.exists(os.path.join(out_dir, "cva_only", "per_class_metrics.csv"))

for i in range(NUM_FOLDS):
    train = f"{KFOLD_DIR}/fold_{i}_train.conll"
    ev = f"{KFOLD_DIR}/fold_{i}_eval.conll"
    out = f"{CVA_ROOT}/fold_{i}"
    print(f"\n========== CVA FOLD {i}/{NUM_FOLDS-1} ==========")
    if not cva_already_done(out):
        run(["python3", "scripts/compare_context_cva.py",
             "--train-file", train, "--eval-file", ev,
             "--output-dir", out, "--device", "cuda"])
    else:
        print("skip (already done):", out)

print("\nCVA tüm foldlar tamamlandı.")


========== CVA FOLD 0/3 ==========
>>> python3 scripts/compare_context_cva.py --train-file data/kfold_4/fold_0_train.conll --eval-file data/kfold_4/fold_0_eval.conll --output-dir /content/drive/MyDrive/ener_kfold_results/kfold_4/cva/fold_0 --device cuda
Comparison completed. Results saved to /content/drive/MyDrive/ener_kfold_results/kfold_4/cva/fold_0


========== CVA FOLD 1/3 ==========
>>> python3 scripts/compare_context_cva.py --train-file data/kfold_4/fold_1_train.conll --eval-file data/kfold_4/fold_1_eval.conll --output-dir /content/drive/MyDrive/ener_kfold_results/kfold_4/cva/fold_1 --device cuda
Comparison completed. Results saved to /content/drive/MyDrive/ener_kfold_results/kfold_4/cva/fold_1


========== CVA FOLD 2/3 ==========
>>> python3 scripts/compare_context_cva.py --train-file data/kfold_4/fold_2_train.conll --eval-file data/kfold_4/fold_2_eval.conll --output-dir /content/drive/MyDrive/ener_kfold_results/kfold_4/cva/fold_2 --device cuda
Comparison completed. Results sav

In [ ]:
import os, zipfile

SRC = "/content/drive/MyDrive/ener_kfold_results/kfold_4/cva"
OUT_ZIP = "/content/cva_4fold_metrics.zip"

KEEP_SUFFIXES = (
    "per_class_metrics.csv", "confusion_matrix.csv", "confusion_matrix_top20.csv",
    "classification_report.json", "comparison_summary.json",
)

count = 0
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(SRC):
        for f in files:
            if f.endswith(KEEP_SUFFIXES):
                zf.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), SRC))
                count += 1

print(f"{count} dosya eklendi")
from google.colab import files
files.download(OUT_ZIP)

52 dosya eklendi


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 8) Sonuçları zip'le ve indir (Drive'da zaten yedekli duruyor)

In [ ]:
import shutil
zip_base = f"/content/ener_kfold_{NUM_FOLDS}fold_results"
shutil.make_archive(zip_base, 'zip', f"/content/drive/MyDrive/ener_kfold_results")
from google.colab import files
files.download(zip_base + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Notlar
- **Süre tahmini**: T4 GPU'da BERT-base fine-tuning, ~800 cümle × 3–5 epoch için fold başına genelde birkaç dakika sürer; 4 model × 4 fold toplamda kabaca 30-90 dakika arası olabilir (donanıma/kuyruğa göre değişir — bu bir tahmin, garanti değil).
- **Colab oturumu koparsa**: Drive'a bağlı olduğun için hücre 7'yi tekrar çalıştırman yeterli, tamamlanan foldlar otomatik atlanır.
- Sonuçları (zip dosyası veya doğrudan Drive klasörü) bana geri gönder, CRF için yaptığım gibi per-label tabloları ve sparse confusion matrix'leri üretip bildiriye/teze işleyeyim.

In [ ]:
import os, zipfile

SRC = "/content/drive/MyDrive/ener_kfold_results"
OUT_ZIP = "/content/ener_kfold_metrics_only.zip"

# sadece küçük metrik/rapor dosyalarını al — model ağırlıklarını (.pt, .bin, bert_backbone/) dışarıda bırak
KEEP_SUFFIXES = (
    "per_class_metrics.csv", "per_label_report.csv",
    "confusion_matrix.csv", "confusion_matrix_top20.csv",
    "classification_report.json", "metrics.json", "summary.json",
    "cv_summary.json", "crf_cv_summary.json",
)

count = 0
total_size = 0
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(SRC):
        # model klasörlerine hiç girme (zaten büyük olan kısım burası)
        dirs[:] = [d for d in dirs if d not in ("bert_backbone", "model")]
        for f in files:
            if f.endswith(KEEP_SUFFIXES):
                full = os.path.join(root, f)
                rel = os.path.relpath(full, SRC)
                zf.write(full, rel)
                total_size += os.path.getsize(full)
                count += 1

print(f"{count} dosya eklendi, toplam boyut: {total_size/1024:.1f} KB")
print("Zip:", OUT_ZIP)

from google.colab import files
files.download(OUT_ZIP)

64 dosya eklendi, toplam boyut: 2277.1 KB
Zip: /content/ener_kfold_metrics_only.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>